# 训练 · 打分 · 回测 Pipeline（XGB + Transformer + Ensemble）

- **与 `main.ipynb` 的关系**：二者都合理——`main.ipynb` 更接近「全历史滚动、默认落盘到 `outputs/scores/`」的**规范实盘前**流程；本 notebook 侧重 **OOS 严格冻结 + 测试对比**（模型与打分写入 `config.PIPELINE_HOLDOUT_*`，不会覆盖 main 的 `SCORE_xgb_*.fea` / `rolling_model.pkl` 等）。
- **训练**：仅 `train ∪ val`（`TRADE_DATE < OOS_START`），与 [main.ipynb](main.ipynb) 中纯 OOS 冻结思路一致。
- **分别输出**：XGB 与 Transformer 各自的 fold IC、全样本 IC 报告、快速分层回测图（含组内累计收益曲线）。
- **Ensemble**：对两路 `full` 打分（样本内+OOS 拼接）做截面 Z-Score 后简单平均，再跑同样报告（实现见 [ensemble_scorer.py](model/ensemble_scorer.py)，与 main 中用法一致）。
- **精细化回测**：可对三份 `score_full` 分别跑 `BacktestRunner`（可选）。

In [ ]:
import sys
import logging
from pathlib import Path

ROOT = Path("/root/autodl-tmp")
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.INFO, force=True)
for name in ("Strategy", "Strategy.model", "Strategy.backtest"):
    logging.getLogger(name).setLevel(logging.INFO)

import pandas as pd
import numpy as np

from Strategy import config

## 参数

- `TRAIN_TRANSFORMER`：若只想先跑通 XGB，可改为 `False`。
- **产出目录**：`config.PIPELINE_HOLDOUT_SCORE_DIR`（滚动模型 pkl、各 `SCORE_*.fea`）、`COMPARE_ROOT`（本格基于 `PIPELINE_HOLDOUT_BT_DIR` 生成，放 IC/quick/事件回测图）。与 main 默认的 `SCORE_OUTPUT_DIR` 根目录隔离。
- `BT_PRICE_TAG`：与 `outputs/labels/{BT_PRICE_TAG}.fea` 成交价宽表一致。

In [ ]:
LABEL_TAG = "OPEN0935_1000"
BT_PRICE_TAG = "OPEN0935_1000"

TRAIN_TRANSFORMER = True

config.PIPELINE_HOLDOUT_SCORE_DIR.mkdir(parents=True, exist_ok=True)
config.PIPELINE_HOLDOUT_BT_DIR.mkdir(parents=True, exist_ok=True)

COMPARE_ROOT = config.PIPELINE_HOLDOUT_BT_DIR / f"compare_{LABEL_TAG}"
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)

TOP_KS = (20, 50, 100)
TAIL_KS = (20, 50, 100)

print("OOS_START =", config.OOS_START)
print("Pipeline 打分目录:", config.PIPELINE_HOLDOUT_SCORE_DIR)
print("对比报告目录:", COMPARE_ROOT)

## 1. 数据：全量 Panel + 切分（训练不含 OOS）

In [ ]:
from Strategy.factor.factor_base import load_all_factors
from Strategy.label.label_generator import load_label
from Strategy.model.trainer import build_panel, split_panel

factor_dict = load_all_factors()
label_df = load_label(LABEL_TAG)

panel = build_panel(factor_dict, label_df)
train, val, oos = split_panel(panel)

panel_in_sample = pd.concat([train, val], axis=0, ignore_index=True)
panel_in_sample = panel_in_sample.sort_values(["TRADE_DATE", "StockID"]).reset_index(drop=True)

oos_start = pd.Timestamp(config.OOS_START).normalize()
assert pd.to_datetime(panel_in_sample["TRADE_DATE"]).max().normalize() < oos_start
assert pd.to_datetime(oos["TRADE_DATE"]).min().normalize() >= oos_start

print("panel:", panel.shape, "| in_sample:", panel_in_sample.shape, "| oos:", oos.shape)

## 2a. XGBoost 滚动训练（仅 in-sample）

In [ ]:
from Strategy.model.rolling_trainer import RollingTrainer
from Strategy.model.trainer import XGBTrainer

rt_xgb = RollingTrainer(
    model_class=XGBTrainer,
    model_kwargs={"use_wpcc": True, "label_winsorize_sigma": 3.0},
    train_start="2021-01-01",
    first_train_months=9,
    val_months=3,
)
rt_xgb.train_all_folds(panel_in_sample)

print("\n========== XGB fold IC ==========")
print(rt_xgb.fold_ic_report())
fi = rt_xgb.get_feature_importance()
if fi is not None:
    print(fi.head(10))

rt_xgb.save_model(config.PIPELINE_HOLDOUT_SCORE_DIR / f"rolling_xgb_{LABEL_TAG}.pkl")

## 2b. Transformer 滚动训练（仅 in-sample）

关闭可将 `TRAIN_TRANSFORMER=False`，后续 ensemble 将跳过（仅保留单模型报告）。

In [ ]:
rt_tf = None
if TRAIN_TRANSFORMER:
    from Strategy.model.transformer_trainer import TransformerTrainer

    rt_tf = RollingTrainer(
        model_class=TransformerTrainer,
        model_kwargs={
            "d_model": 64, "nhead": 4, "num_layers": 2, "d_ff": 128, "dropout": 0.15,
            "epochs": 50, "lr": 5e-4, "weight_decay": 0.01, "batch_size": 16,
            "early_stopping_patience": 8, "device": "cuda", "use_amp": True,
            "label_winsorize_sigma": 3.0,
        },
        train_start="2021-01-01",
        first_train_months=9,
        val_months=3,
    )
    rt_tf.train_all_folds(panel_in_sample)
    print("\n========== Transformer fold IC ==========")
    print(rt_tf.fold_ic_report())
    rt_tf.save_model(config.PIPELINE_HOLDOUT_SCORE_DIR / f"rolling_transformer_{LABEL_TAG}.pkl")
else:
    print("已跳过 Transformer 训练 (TRAIN_TRANSFORMER=False)")

## 3. 打分：各模型 in-sample / OOS / full，并落盘

`full` = 样本内与 OOS 按日期拼接；文件落在 **`PIPELINE_HOLDOUT_SCORE_DIR`**，不会与 main 根目录下的 `load_scores("xgb", ...)` 默认路径冲突。读取时用 `pd.read_feather(...).set_index("TRADE_DATE")` 或自行封装路径。

In [ ]:
from Strategy.model.scorer import mask_scores_by_current_price
from Strategy.data_io.saver import save_wide_table

def _score_and_save(rt, tag: str):
    s_is = mask_scores_by_current_price(rt.score_all(panel_in_sample, normalize=True), label_tag=LABEL_TAG)
    s_oos = mask_scores_by_current_price(rt.score_all(oos, normalize=True), label_tag=LABEL_TAG)
    s_full = pd.concat([s_is, s_oos]).sort_index()
    s_full = s_full[~s_full.index.duplicated(keep="last")]
    d = config.PIPELINE_HOLDOUT_SCORE_DIR
    save_wide_table(s_is, d / f"SCORE_{tag}_{LABEL_TAG}_in_sample.fea")
    save_wide_table(s_oos, d / f"SCORE_{tag}_{LABEL_TAG}_oos.fea")
    save_wide_table(s_full, d / f"SCORE_{tag}_{LABEL_TAG}.fea")
    return s_is, s_oos, s_full

score_xgb_is, score_xgb_oos, score_xgb_full = _score_and_save(rt_xgb, "xgb")
print("XGB shapes:", score_xgb_is.shape, score_xgb_oos.shape, score_xgb_full.shape)

score_tf_is = score_tf_oos = score_tf_full = None
if rt_tf is not None:
    score_tf_is, score_tf_oos, score_tf_full = _score_and_save(rt_tf, "transformer")
    print("Transformer shapes:", score_tf_is.shape, score_tf_oos.shape, score_tf_full.shape)

## 4. Ensemble 打分（Z-Score 简单平均）

逻辑与 main 中 `ensemble_scorer` 相同；落盘为 **`PIPELINE_HOLDOUT_SCORE_DIR / SCORE_ensemble_{LABEL_TAG}.fea`**（main 默认仍写 `SCORE_OUTPUT_DIR`，二者互不覆盖）。

In [ ]:
from Strategy.model.ensemble_scorer import compute_score_correlation, select_ensemble_models, ensemble_scores

score_ensemble_full = None
if rt_tf is not None:
    dfs = {"xgb": score_xgb_full, "transformer": score_tf_full}
    print("\n========== 模型打分截面平均相关性 ==========")
    print(compute_score_correlation(dfs))
    selected = select_ensemble_models(dfs, ic_summaries=None, max_pairwise_corr=0.95)
    print("入选集成模型:", selected)
    score_ensemble_full = ensemble_scores(
        dfs, selected_models=selected, label_tag=LABEL_TAG, save=False
    )
    score_ensemble_full = mask_scores_by_current_price(score_ensemble_full, label_tag=LABEL_TAG)
    save_wide_table(score_ensemble_full, config.PIPELINE_HOLDOUT_SCORE_DIR / f"SCORE_ensemble_{LABEL_TAG}.fea")
    print("ensemble full:", score_ensemble_full.shape)
else:
    print("无 Transformer 打分，跳过 ensemble")

## 5. 对比报告：IC / Rank IC（分段统计 + 累计 IC 图）+ 快速分层（累计收益曲线）

对 **xgb / transformer / ensemble** 各输出一套子目录（与 main 中 `models_to_test` 循环类似）：

- `ic_analysis.png`、`ic_series.csv`、`ic_summary.csv` —— [ic_analysis.run_ic_analysis](backtest/ic_analysis.py)
- `quantile_backtest.png`、`topN_tailN_nav.png` —— [quick_backtest.run_quick_backtest](backtest/quick_backtest.py)（`run_ic=False` 避免与上重复）

区间：`VAL_START` 起至数据末（含纯 OOS）。另可对 **仅 OOS** 再跑一版 quick（见下一格）。

In [ ]:
from Strategy.backtest.ic_analysis import run_ic_analysis
from Strategy.backtest.quick_backtest import run_quick_backtest

label_wide = load_label(LABEL_TAG)

def report_one(name: str, score_full: pd.DataFrame):
    if score_full is None or len(score_full) == 0:
        print(f"跳过 {name}: 无打分")
        return
    base = COMPARE_ROOT / name
    base.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*20} {name} | IC 分析 {'='*20}")
    ic_dir = base / "ic_val_to_end"
    ic_df, summary = run_ic_analysis(
        score_full,
        label_wide,
        title=f"{name.upper()} | {LABEL_TAG} | IC",
        output_dir=ic_dir,
        rolling_window=20,
    )
    print(summary)
    print(f"\n--- {name} | quick 分层 (VAL->末, 含OOS) ---")
    run_quick_backtest(
        score_full,
        label_wide,
        n_groups=config.N_QUANTILE_GROUPS,
        title=f"{name.upper()} | {LABEL_TAG} | quick",
        output_dir=base / "quick_val_to_end",
        start_date=config.VAL_START,
        end_date=None,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        twap_tag=BT_PRICE_TAG,
        run_ic=False,
    )

report_one("xgb", score_xgb_full)
report_one("transformer", score_tf_full)
report_one("ensemble", score_ensemble_full)

## 5b. 仅纯 OOS 区间：快速分层（便于对照样本外累计收益）

In [ ]:
def quick_oos_only(name: str, score_oos: pd.DataFrame):
    if score_oos is None or len(score_oos) == 0:
        return
    out = COMPARE_ROOT / name / "quick_oos_only"
    run_quick_backtest(
        score_oos,
        label_wide,
        n_groups=config.N_QUANTILE_GROUPS,
        title=f"{name.upper()} | {LABEL_TAG} | OOS",
        output_dir=out,
        start_date=config.OOS_START,
        end_date=None,
        top_ks=TOP_KS,
        tail_ks=TAIL_KS,
        twap_tag=BT_PRICE_TAG,
        run_ic=False,
    )

score_ens_oos = None
if score_ensemble_full is not None:
    d_oos = pd.to_datetime(score_ensemble_full.index).normalize() >= oos_start
    score_ens_oos = score_ensemble_full.loc[d_oos]

quick_oos_only("xgb", score_xgb_oos)
quick_oos_only("transformer", score_tf_oos)
quick_oos_only("ensemble", score_ens_oos)

## 6. 精细化事件回测（各模型 `score_full`，VAL 起至数据末）

与 main 中注释的 `BacktestRunner` 用法一致；`split_date` 标出 OOS 起点。

In [ ]:
from Strategy.backtest.event_backtest import BacktestRunner

def event_one(name: str, score_full: pd.DataFrame):
    if score_full is None or len(score_full) == 0:
        return
    evd = COMPARE_ROOT / name / "event_val_to_end"
    evd.mkdir(parents=True, exist_ok=True)
    runner = BacktestRunner(
        score_df=score_full,
        top_n=50,
        n_quantile_groups=config.N_QUANTILE_GROUPS,
        rebalance_freq=1,
        initial_capital=config.INITIAL_CAPITAL,
        frictionless=False,
        twap_tag=BT_PRICE_TAG,
    )
    res = runner.run(start_date=config.VAL_START, end_date=None)
    res.plot(save_dir=evd, split_date=config.OOS_START)
    res.save_details(evd)
    print(f"事件回测已写入: {evd}")

event_one("xgb", score_xgb_full)
event_one("transformer", score_tf_full)
event_one("ensemble", score_ensemble_full)